In [0]:
import yaml
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.profiler.generator import DQGenerator
from pyspark.sql import functions as F

ws = WorkspaceClient()

file_path = "/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_data_contract/test2.yml"

with open(file_path, "r") as f:
    contract = yaml.safe_load(f)

# Generate DQX rules (including AI-assisted text rules)
generator = DQGenerator(workspace_client=ws)
rules = generator.generate_rules_from_contract(
    contract_file=file_path,
    process_text_rules=False,  # Disable AI-assisted rule generation to avoid endpoint requirement
    default_criticality="error"  # or "warn" for warnings instead of errors
)

#print(f"Generated {len(rules)} quality rules from contract")

# 1. Load the raw JSON file
raw_df = spark.read.option("multiLine", "true").json("/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_property_unit_of_property_changeset/property_unit_of_property_changeset_5_2026-05-04_21-11-55.json")

# 2. Flatten the 'features' array so each feature is a row
# This extracts the 'properties', 'geometry', and 'id' into top-level columns
df = raw_df.select(F.explode("features").alias("feature")) \
    .select(
        "feature.id",
        "feature.geometry.type",
        "feature.geometry.coordinates",
        "feature.properties.*" # Flattens all keys inside 'properties'
    )

display(df)

In [0]:
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
generator = DQGenerator(workspace_client=ws, spark=spark)

# Generate rules including text-based expectations (requires LLM)
rules = generator.generate_rules_from_contract(
    contract_file="/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_data_contract/linz_data_contract_with_sourcepath_dqx.yml",
    generate_predefined_rules=True,   # Include schema-derived rules
    process_text_rules=True         # Process text expectations with LLM
)
print(rules)
# The generated rules will include:
# - Predefined rules from schema constraints
# - Explicit DQX rules (if any)
# - Rules generated from text expectations via LLM

In [0]:
pip install databricks-labs-dqx

In [0]:
import yaml
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.profiler.generator import DQGenerator
from pyspark.sql import functions as F

ws = WorkspaceClient()

file_path = "/Volumes/places/enterprise_steady_state/ess0_linz_json/linz_data_contract/linz_data_contract_prototype.yml"

with open(file_path, "r") as f:
    contract = yaml.safe_load(f)

schema_list = []
    
# Iterate through each schema defined in the file
for item in contract.get('schema', []):
        # Extract the requested fields at the same level
        schema_details = {
            'name': item.get('name'),
            'contract_path': item.get('contract_path'),
            'source': item.get('source'),
            'volume': item.get('volume'),
            'table_name': item.get('table_name'),
            'folder': item.get('folder'),
            'schema': item.get('schema'),
            'key': item.get('key'),
            'properties': [p.get('name') for p in item.get('properties', [])]
        }
        schema_list.append(schema_details)

print(schema_list)

